# JEPA Lambda Sweep

Tune LAMBDA_JEPA to find the sweet spot. Separate checkpoints/logs from jepa_train.

In [ ]:
# Environment setup — works on both Colab and local
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Clone repo if not already present
    REPO_DIR = '/content/drive/MyDrive/Symba/symbolic-jepa'
    if not os.path.exists(REPO_DIR):
        %cd /content/drive/MyDrive/Symba
        !git clone https://github.com/zzpDavid2/symbolic-jepa.git {REPO_DIR}
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

    !pip install -q sympy scipy

    # Checkpoint dir on Drive for persistence
    CKPT_DIR = '/content/drive/MyDrive/Symba/symbolic-jepa/checkpoints/jepa_sweep'
    LOG_DIR  = '/content/drive/MyDrive/Symba/symbolic-jepa/runs'
else:
    CKPT_DIR = 'checkpoints/jepa_sweep'
    LOG_DIR  = 'runs'

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
print(f'Environment: {"Colab" if IN_COLAB else "Local"}')
print(f'Checkpoints: {CKPT_DIR}')

In [53]:
if IN_COLAB:
    %cd /content/drive/MyDrive/Symba/symbolic-jepa
    !git pull

In [ ]:
import gc
import torch
import torch.nn.functional as F
import numpy as np
import random
import datetime
import json
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from symbolic_jepa import (
    PrefixTokenizer, Expression,
    TNet, SymbolicTransformer,
    JEPAPredictor, IdentityPredictor, jepa_loss,
    PointCloudDataset, build_synthetic_splits,
    load_synthetic_pkl,
    teacher_forced_accuracy, teacher_forced_counts,
    evaluate_predictions, cleanup_eval_pool,
    sym_spread, pred_spread, retrieval_top1, common_mode,
)
from symbolic_jepa.tokenizer import prefix_to_sympy

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# ── Hyperparameters ──
MAX_VARS    = 1               # univariate synthetic data
D_INPUT     = MAX_VARS + 1    # (x, y) = 2
N_POINTS    = 1000
MAX_SEQ     = 64
D_MODEL     = 512
N_HEADS     = 8
N_LAYERS    = 4
DROPOUT     = 0.2

EPOCHS      = 30
LR          = 3e-4
BATCH       = 16
VAL_EVERY   = 1
USE_AMP     = True

# JEPA alignment
JEPA_LOSS      = 'cosine'       # 'cosine' (ordinary) | 'centered'
JEPA_STOPGRAD  = False          # gradients flow through z_sym
JEPA_PREDICTOR = 'identity'     # 'identity' | 'mlp'

# ── Training strategy ──
#   'joint'    : L = CE + lambda * JEPA throughout (the original experiment)
#   'pretrain' : Stage 1 = JEPA only, then Stage 2 = CE only
TRAINING_MODE = 'joint'
assert TRAINING_MODE in {'joint', 'pretrain'}, (
    f"TRAINING_MODE must be 'joint' or 'pretrain', got {TRAINING_MODE!r}")

JEPA_PRETRAIN_EPOCHS = 5       # Stage 1 length; ignored when mode == 'joint'
FINETUNE_EPOCHS      = EPOCHS  # Stage 2 length == the original EPOCHS

# Stop-gradient on z_sym during STAGE 1 ONLY.  Joint mode still uses
# JEPA_STOPGRAD above, unchanged.  This is NOT cosmetic -- without it the
# JEPA-only stage collapses; see the note under "JEPA-only collapse".
# Set False to reproduce the collapse.
PRETRAIN_STOPGRAD = True

# Sweep config
LAMBDA_VALUES = [0, 0.01, 0.03, 0.1, 0.3]
SEEDS         = [42, 123, 7]
VERSION_TAG   = 'jepa_identity_v3'

# Synthetic data (pre-generated by SYMBA_Reg_Data_Gen notebook)
SYNTH_PKL   = 'data/synthetic.pkl'
MAX_SYNTH   = 10_000
SYNTH_SEED  = 42

### JEPA-only collapse — why `PRETRAIN_STOPGRAD` exists

Stage 1 optimises the JEPA objective alone. With the sweep's default
`JEPA_STOPGRAD = False`, **both** sides of the comparison are trainable
(`jepa_loss` gradients reach `encoder`, `tok_embed`, `pos_embed`,
`transformer`, `norm` — and `head.weight` is tied to `tok_embed.weight`).
In joint mode CE anchors both sides. Remove CE and the pair is free to
converge on a single constant vector: `cos = 1`, loss = 0, nothing learned.

Measured on a reduced config (d_model=128, 458 equations, 6 JEPA-only epochs):

| | `stopgrad=False` | `stopgrad=True` |
|---|---|---|
| JEPA loss | 1.028 → **0.0037** | 1.028 → 0.314 |
| std(z_sym) | 0.716 → **0.078** | 0.716 → 0.716 (frozen) |
| off-diag cos(z_sym) | 0.473 → **0.994** | 0.473 → 0.473 (frozen) |
| std(z_num) | 0.011 → 0.298 | 0.011 → **0.370** |
| retrieval@1 | ~chance | rises above chance |

`stopgrad=False` shows the full collapse signature: the symbolic side moves
to meet the numeric side, every equation lands on nearly the same point, and
the loss is "solved" without learning anything.

**Minimum change made:** apply stop-gradient to `z_sym` during Stage 1 only.
This reuses `encode_expression` and `jepa_loss` exactly as they already are —
no EMA encoder, no new predictor, no BYOL mechanism. Joint mode is untouched
and still honours `JEPA_STOPGRAD`.

**Consequence to keep in mind when reading results:** with the stop-gradient,
Stage 1 trains **only the T-Net encoder**, against a fixed (randomly
initialised) symbolic target. The decoder is not pretrained. So this tests
"does JEPA-pretraining the point-cloud encoder help?", not "does pretraining
the whole model help?". Set `PRETRAIN_STOPGRAD = False` to reproduce the
collapse above.

## Load synthetic training data

In [55]:
tokenizer = PrefixTokenizer(max_vars=MAX_VARS)
print(f'Vocab size: {len(tokenizer)}')

print(f'Loading synthetic expressions from {SYNTH_PKL}...')
synth_exprs = load_synthetic_pkl(
    SYNTH_PKL, max_seq_len=MAX_SEQ,
    tokenizer=tokenizer, max_expressions=MAX_SYNTH,
)
print(f'Loaded {len(synth_exprs)} expressions')

# Inspect a few
for expr in synth_exprs[:5]:
    print(f'  {expr.prefix}')

Vocab size: 27
Loading synthetic expressions from data/synthetic.pkl...
Loaded 9610 expressions
  mul mul C log add C pow add add add add C mul C x1 mul C pow x1 C mul C pow x1 C mul C pow x1 two two tanh add C mul C x1
  add mul mul C exp mul C pow x1 two sinh mul C x1 mul mul C pow add C mul C x1 neg1 sin add C mul C x1
  mul mul mul C cosh mul C x1 exp mul C pow x1 two exp mul C pow add C x1 two
  add mul mul C pow add C pow add C mul C x1 two neg1 add C mul C x1 mul mul C cosh mul C x1 exp mul C pow x1 two
  mul mul C pow add C mul C x1 neg1 sin add C mul C x1


/Users/zzpdavid2/Documents/GitHub/symbolic-jepa/symbolic_jepa/expressions.py:458: UserWarning: Skipped 390/10000 expressions (parse=0, tokenize=0, too_long=390, unk=0). Examples: 
  warnings.warn(


In [56]:
# Split synthetic data
synth_train, synth_val, synth_test = build_synthetic_splits(
    synth_exprs, tokenizer,
    n_points=N_POINTS, max_seq_len=MAX_SEQ, max_vars=MAX_VARS,
    seed=SYNTH_SEED,
)

Synthetic train: 7688 equations
Synthetic val: 961 equations
Synthetic test: 961 equations


## Build model (inside sweep loop)

Model + predictor are built fresh per (lambda, seed) inside `run_one()`.

## Training loop

In [ ]:
from torch.utils.tensorboard import SummaryWriter

In [ ]:
import time

def seed_everything(seed):
    """Full re-seed for reproducibility."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed + worker_id)
    random.seed(worker_seed + worker_id)

def _recover_best_val_acc(ck_path):
    """Recover best-epoch val_acc from old checkpoints that lack best_val_acc.

    Finds the epoch with min val loss in history and returns its val_acc.
    """
    ck = torch.load(ck_path, map_location='cpu', weights_only=False)
    history = ck.get('history', {})
    val_losses = history.get('val', [])
    val_accs = history.get('val_acc', [])
    if not val_losses or not val_accs:
        return 0.0
    n = min(len(val_losses), len(val_accs))
    best_idx = int(np.argmin(val_losses[:n]))
    return val_accs[best_idx]

def make_run_tag(lam, seed):
    """Run tag. Joint keeps the original scheme so existing checkpoints
    still resolve; pretrain gets its own prefix so the two modes can never
    overwrite each other."""
    if TRAINING_MODE == 'pretrain':
        return f'pretrain_p{JEPA_PRETRAIN_EPOCHS}_lam{lam}_seed{seed}'
    return f'lam{lam}_seed{seed}'


def train_one(lam, seed, synth_train, synth_val, tokenizer):
    """Train one (lambda, seed) run. No eval — just training + checkpoints."""
    seed_everything(seed)

    run_tag = make_run_tag(lam, seed)
    run_dir = f'{CKPT_DIR}/{VERSION_TAG}/{run_tag}'
    CKPT_PATH = f'{run_dir}/latest.pt'
    BEST_PATH = f'{run_dir}/best.pt'

    # Skip if training already completed (latest.pt exists with final epoch)
    if os.path.exists(CKPT_PATH):
        ck = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
        if ck['epoch'] >= EPOCHS:
            print(f'\n{run_tag}: training complete (epoch {ck["epoch"]}), SKIPPING')
            return
    os.makedirs(run_dir, exist_ok=True)

    print(f'\n{"="*60}')
    print(f'{run_tag} | mode={TRAINING_MODE} | loss={JEPA_LOSS} | '
          f'pred={JEPA_PREDICTOR} | stopgrad={JEPA_STOPGRAD} | tag={VERSION_TAG}')
    print(f'{"="*60}')

    encoder = TNet(d_input=D_INPUT, d_model=D_MODEL)
    model = SymbolicTransformer(
        encoder=encoder, vocab_size=len(tokenizer),
        d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
        d_ff=4 * D_MODEL, max_seq_len=MAX_SEQ,
        dropout=DROPOUT, pad_id=tokenizer.pad_id,
    ).to(DEVICE)

    predictor = None
    if lam > 0 or TRAINING_MODE == 'pretrain':
        if JEPA_PREDICTOR == 'mlp':
            predictor = JEPAPredictor(D_MODEL).to(DEVICE)
        else:
            predictor = IdentityPredictor().to(DEVICE)

    g = torch.Generator()
    g.manual_seed(seed)
    train_loader = DataLoader(synth_train, batch_size=BATCH, shuffle=True,
                              num_workers=2, persistent_workers=True, pin_memory=True,
                              worker_init_fn=seed_worker, generator=g)
    val_loader = DataLoader(synth_val, batch_size=BATCH, shuffle=False,
                            num_workers=2, persistent_workers=True,
                            worker_init_fn=seed_worker)

    params = list(model.parameters())
    if predictor is not None:
        params += list(predictor.parameters())
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    start_epoch = 1
    best_val = float('inf')
    best_val_acc = 0.0
    history = {'train': [], 'val': []}

    if os.path.exists(CKPT_PATH):
        ck = torch.load(CKPT_PATH, map_location=DEVICE)
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler'])
        start_epoch = ck['epoch'] + 1
        best_val = ck['best_val']
        best_val_acc = ck.get('best_val_acc', 0.0)
        history = ck['history']
        if predictor is not None and 'predictor' in ck:
            predictor.load_state_dict(ck['predictor'])
        print(f'  Resuming from epoch {start_epoch} (best val: {best_val:.4f})')

    if USE_AMP and DEVICE == 'cuda':
        amp_ctx = lambda: torch.autocast('cuda', dtype=torch.bfloat16)
    elif USE_AMP and DEVICE == 'mps':
        amp_ctx = lambda: torch.autocast('mps', dtype=torch.float16)
    else:
        amp_ctx = lambda: torch.amp.autocast('cpu', enabled=False)

    tb_dir = f'{LOG_DIR}/{VERSION_TAG}/{run_tag}'
    writer = SummaryWriter(log_dir=tb_dir)

    # ══════════════════════════════════════════════════════════════
    # Stage 1: JEPA-only pretraining  (TRAINING_MODE == 'pretrain')
    # ══════════════════════════════════════════════════════════════
    # Gradient flow here is JEPA -> encoder only (with PRETRAIN_STOPGRAD=True).
    # There is deliberately NO decoder forward pass and NO CE term: the loss
    # is the bare JEPA objective, not lam * jepa, so `lam` is not the
    # experimental variable in this mode.
    #
    # Skipped when resuming (start_epoch > 1): Stage 1 already ran and its
    # weights are inside the checkpoint we just loaded.
    if TRAINING_MODE == 'pretrain' and start_epoch == 1:
        print(f'\n=== Stage 1: JEPA pretraining '
              f'({JEPA_PRETRAIN_EPOCHS} epochs, loss = JEPA only) ===')
        print(f'  stopgrad={PRETRAIN_STOPGRAD}  '
              f'trains={"encoder only" if PRETRAIN_STOPGRAD else "encoder + decoder"}')
        # Stage 1 gets its OWN optimizer; a fresh one is built for Stage 2 so
        # no Adam moments / LR schedule carry across the objective switch.
        pre_opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.1)

        for pe in range(1, JEPA_PRETRAIN_EPOCHS + 1):
            model.train()
            if predictor is not None:
                predictor.train()
            pre_sum = pre_n = 0
            std_num_sum = std_sym_sum = 0.0
            pbar_p = tqdm(train_loader, leave=False,
                          desc=f'{run_tag} P{pe}/{JEPA_PRETRAIN_EPOCHS}')
            for batch in pbar_p:
                points    = batch['points'].to(DEVICE, non_blocking=True)
                input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
                attn_mask = batch['attn_mask'].to(DEVICE, non_blocking=True)

                pre_opt.zero_grad()
                with amp_ctx():
                    # Encoder only -- no decoder forward, so no CE can leak in.
                    z_num = model.encoder(points)
                    if PRETRAIN_STOPGRAD:
                        with torch.no_grad():
                            z_sym = model.encode_expression(
                                input_ids, attn_mask=attn_mask)
                    else:
                        z_sym = model.encode_expression(
                            input_ids, attn_mask=attn_mask)
                    loss_pre = jepa_loss(predictor(z_num), z_sym, mode=JEPA_LOSS)

                loss_pre.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                if predictor is not None:
                    torch.nn.utils.clip_grad_norm_(predictor.parameters(), 1.0)
                pre_opt.step()

                pre_sum += loss_pre.item(); pre_n += 1
                # Averaged over the whole epoch, not just the last batch:
                # a per-batch reading jitters with which equations land in
                # the final batch, which would mask a real collapse trend.
                std_num_sum += z_num.detach().float().std(dim=0).mean().item()
                std_sym_sum += z_sym.detach().float().std(dim=0).mean().item()
                pbar_p.set_postfix({'jepa': f'{loss_pre.item():.4f}'})
            pbar_p.close(); del pbar_p

            # Collapse diagnostics: mean per-dim std of the embeddings.
            # Trending to 0 means the representation is degenerating to a
            # constant. With PRETRAIN_STOPGRAD=True, std(z_sym) should stay
            # flat -- the symbolic side is frozen.
            pre_avg = pre_sum / max(pre_n, 1)
            std_num = std_num_sum / max(pre_n, 1)
            std_sym = std_sym_sum / max(pre_n, 1)
            history.setdefault('pretrain_jepa', []).append(pre_avg)
            history.setdefault('pretrain_std_num', []).append(std_num)
            history.setdefault('pretrain_std_sym', []).append(std_sym)
            writer.add_scalar('pretrain/jepa_loss', pre_avg, pe)
            writer.add_scalar('pretrain/std_z_num', std_num, pe)
            writer.add_scalar('pretrain/std_z_sym', std_sym, pe)
            print(f'  P{pe}/{JEPA_PRETRAIN_EPOCHS} | jepa={pre_avg:.4f} | '
                  f'std(z_num)={std_num:.4f} std(z_sym)={std_sym:.4f}')

        # ── Stage transition: keep weights, drop optimizer/scheduler state ──
        del pre_opt
        optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.1)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS)
        print(f'\n=== Stage 2: CE fine-tuning '
              f'({EPOCHS} epochs, loss = CE only) ===')

    # In pretrain mode the loop below IS Stage 2, so JEPA must contribute no
    # gradient: lam_eff forces the auxiliary term off. Joint mode is unchanged.
    lam_eff = 0.0 if TRAINING_MODE == 'pretrain' else lam

    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        if predictor is not None:
            predictor.train()
        train_loss_gen = 0
        pbar = tqdm(train_loader, desc=f'{run_tag} E{epoch}/{EPOCHS}', leave=False)
        for batch in pbar:
            points    = batch['points'].to(DEVICE, non_blocking=True)
            input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
            attn_mask = batch['attn_mask'].to(DEVICE, non_blocking=True)

            optimizer.zero_grad()
            with amp_ctx():
                out = model(points, input_ids, attn_mask=attn_mask)
                loss_gen = out['loss']

                if lam_eff > 0:
                    if JEPA_STOPGRAD:
                        model.eval()
                        with torch.no_grad():
                            z_sym = model.encode_expression(input_ids, attn_mask=attn_mask)
                        model.train()
                    else:
                        z_sym = model.encode_expression(input_ids, attn_mask=attn_mask)

                    z_num = out['z_num']
                    z_pred = predictor(z_num)
                    loss_align = jepa_loss(z_pred, z_sym, mode=JEPA_LOSS)
                    loss = loss_gen + lam_eff * loss_align
                else:
                    loss = loss_gen

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if predictor is not None:
                torch.nn.utils.clip_grad_norm_(predictor.parameters(), 1.0)
            optimizer.step()

            train_loss_gen += loss_gen.item()
            pbar.set_postfix({'loss': f'{loss_gen.item():.4f}'})

            global_step = (epoch - 1) * len(train_loader) + pbar.n
            writer.add_scalar('train/loss_step', loss_gen.item(), global_step)
            if lam_eff > 0:
                writer.add_scalar('train/jepa_raw', loss_align.item(), global_step)
                writer.add_scalar('train/jepa_weighted',
                                  (lam_eff * loss_align).item(), global_step)

        scheduler.step()
        train_avg = train_loss_gen / len(train_loader)
        history['train'].append(train_avg)
        writer.add_scalar('train/loss_epoch', train_avg, epoch)
        writer.add_scalar('train/lr', scheduler.get_last_lr()[0], epoch)

        # ── Validation (token-weighted aggregation) ──
        if epoch % VAL_EVERY == 0 or epoch == EPOCHS:
            model.eval()
            if predictor is not None:
                predictor.eval()
            val_loss_sum = 0.0     # sum of (batch_loss * n_valid_tokens)
            val_tokens_total = 0   # total valid (non-pad) target tokens
            acc_correct = 0.0      # total correct tokens across all batches
            acc_total = 0.0        # total valid tokens for accuracy
            val_align_sum = 0
            n_val_batches = 0
            diag_z_sym = None
            diag_z_pred = None
            with torch.no_grad(), amp_ctx():
                for batch in val_loader:
                    points    = batch['points'].to(DEVICE, non_blocking=True)
                    input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
                    attn_mask = batch['attn_mask'].to(DEVICE, non_blocking=True)
                    out = model(points, input_ids, attn_mask=attn_mask)

                    # Token-weighted loss: weight by number of valid tokens
                    n_tok = out['n_tokens']
                    val_loss_sum += out['loss'].item() * n_tok
                    val_tokens_total += n_tok

                    # Token-weighted accuracy: accumulate counts
                    c, t = teacher_forced_counts(out['logits'], input_ids, tokenizer.pad_id)
                    acc_correct += c
                    acc_total += t

                    if predictor is not None:
                        z_sym_v = model.encode_expression(input_ids, attn_mask=attn_mask)
                        z_num_v = out['z_num']
                        z_pred_v = predictor(z_num_v)
                        val_align_sum += jepa_loss(z_pred_v, z_sym_v, mode=JEPA_LOSS).item()
                        if diag_z_sym is None:
                            diag_z_sym = z_sym_v
                            diag_z_pred = z_pred_v

                    n_val_batches += 1

            val_avg = val_loss_sum / max(val_tokens_total, 1)
            val_acc_avg = acc_correct / max(acc_total, 1)
            history['val'].append(val_avg)
            history.setdefault('val_acc', []).append(val_acc_avg)
            writer.add_scalar('val/loss', val_avg, epoch)
            writer.add_scalar('val/token_accuracy', val_acc_avg, epoch)

            extra = ''
            if predictor is not None:
                val_align_avg = val_align_sum / n_val_batches
                writer.add_scalar('val/jepa_raw', val_align_avg, epoch)
                history.setdefault('val_jepa_raw', []).append(val_align_avg)

                ss = sym_spread(diag_z_sym)
                ps = pred_spread(diag_z_pred)
                rt = retrieval_top1(diag_z_pred, diag_z_sym)
                cm = common_mode(diag_z_sym)

                writer.add_scalar('val/sym_spread_raw', ss['raw'], epoch)
                writer.add_scalar('val/sym_spread_cent', ss['centered'], epoch)
                writer.add_scalar('val/pred_spread', ps['raw'], epoch)
                writer.add_scalar('val/retrieval_top1', rt['centered'], epoch)
                writer.add_scalar('val/common_mode_ratio', cm['mean_norm_ratio'], epoch)

                extra = (f' | jepa={val_align_avg:.4f}'
                         f' | retr={rt["centered"]:.2f}')

            is_best = val_avg < best_val
            if is_best:
                best_val = val_avg
                best_val_acc = val_acc_avg
            flag = ' * best' if is_best else ''
            print(f'  E{epoch}/{EPOCHS} | train={train_avg:.4f} | val={val_avg:.4f}{flag} | acc={val_acc_avg*100:.1f}%{extra}')

            if is_best:
                torch.save({
                    'model': model.state_dict(),
                    'epoch': epoch,
                    'val': val_avg,
                    'val_acc': val_acc_avg,
                }, BEST_PATH)

        ckpt_dict = {
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'epoch': epoch,
            'best_val': best_val,
            'best_val_acc': best_val_acc,
            'history': history,
            'lambda_jepa': lam,
            'training_mode': TRAINING_MODE,
            'jepa_pretrain_epochs': (JEPA_PRETRAIN_EPOCHS
                                     if TRAINING_MODE == 'pretrain' else 0),
            'pretrain_stopgrad': (PRETRAIN_STOPGRAD
                                  if TRAINING_MODE == 'pretrain' else None),
            'seed': seed,
            'jepa_loss_mode': JEPA_LOSS,
            'jepa_predictor': JEPA_PREDICTOR,
            'jepa_stopgrad': JEPA_STOPGRAD,
        }
        if predictor is not None and isinstance(predictor, JEPAPredictor):
            ckpt_dict['predictor'] = predictor.state_dict()
        torch.save(ckpt_dict, CKPT_PATH)

    writer.close()

    del train_loader, val_loader
    del model, encoder, predictor, optimizer, scheduler
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()


def eval_one(lam, seed, synth_test, tokenizer):
    """Load best checkpoint and run greedy eval on full test set. Returns metrics dict."""
    run_tag = make_run_tag(lam, seed)
    run_dir = f'{CKPT_DIR}/{VERSION_TAG}/{run_tag}'
    metrics_path = f'{run_dir}/metrics.json'
    BEST_PATH = f'{run_dir}/best.pt'
    CKPT_PATH = f'{run_dir}/latest.pt'

    # Skip if already evaluated
    if os.path.exists(metrics_path):
        with open(metrics_path) as f:
            return json.load(f)

    # Load best model
    encoder = TNet(d_input=D_INPUT, d_model=D_MODEL)
    model = SymbolicTransformer(
        encoder=encoder, vocab_size=len(tokenizer),
        d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
        d_ff=4 * D_MODEL, max_seq_len=MAX_SEQ,
        dropout=0.0, pad_id=tokenizer.pad_id,
    ).to(DEVICE)

    ck_path = BEST_PATH if os.path.exists(BEST_PATH) else CKPT_PATH
    ck = torch.load(ck_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(ck['model'])
    model.eval()

    # Get best-epoch val_acc:
    # 1. New checkpoints: best.pt has 'val_acc', latest.pt has 'best_val_acc'
    # 2. Old checkpoints: recover from history (find epoch with min val loss)
    best_val_acc = ck.get('val_acc', None)
    if best_val_acc is None:
        latest_ck = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
        best_val_acc = latest_ck.get('best_val_acc', None)
    if best_val_acc is None or best_val_acc == 0.0:
        best_val_acc = _recover_best_val_acc(CKPT_PATH)

    best_val = ck.get('val', None)
    if best_val is None:
        latest_ck = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
        best_val = latest_ck.get('best_val', float('inf'))

    # Greedy decode on FULL test set
    eval_loader = DataLoader(synth_test, batch_size=BATCH, shuffle=False)

    greedy_preds = []
    for batch in tqdm(eval_loader, desc=f'{run_tag} decode', leave=False):
        points = batch['points'].to(DEVICE)
        input_ids = batch['input_ids']
        preds = model.generate(points, tokenizer, max_new_tokens=MAX_SEQ)
        for j, pred_str in enumerate(preds):
            gt_str = tokenizer.decode(input_ids[j].tolist())
            greedy_preds.append((gt_str, pred_str))

    greedy_results = evaluate_predictions(greedy_preds, synth_test, tokenizer)

    metrics = {
        'lambda': lam,
        'seed': seed,
        'run_tag': run_tag,
        'jepa_loss_mode': JEPA_LOSS,
        'jepa_predictor': JEPA_PREDICTOR,
        'jepa_stopgrad': JEPA_STOPGRAD,
        'training_mode': TRAINING_MODE,
        'jepa_pretrain_epochs': (JEPA_PRETRAIN_EPOCHS
                                 if TRAINING_MODE == 'pretrain' else 0),
        'pretrain_stopgrad': (PRETRAIN_STOPGRAD
                              if TRAINING_MODE == 'pretrain' else None),
        'version_tag': VERSION_TAG,
        'best_val_loss': best_val,
        'best_val_acc': best_val_acc,
        'greedy_exact_match': greedy_results['exact_match'],
        'greedy_token_acc': greedy_results['token_accuracy'],
        'greedy_algebraic_equiv': greedy_results['algebraic_equiv'],
        'greedy_r2_above_0.9': greedy_results['r2_above_0.9'],
        'mean_r2': greedy_results['mean_r2'],
        'median_r2': greedy_results['median_r2'],
        'n_parseable': greedy_results['n_parseable'],
        'n_total': greedy_results['n_total'],
        'details': greedy_results['details'],
    }
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)

    del model, encoder, eval_loader
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()

    return metrics


# ══════════════════════════════════════════════════════════════
# Phase 1: Train all runs (no sympy, no stuck threads)
# ══════════════════════════════════════════════════════════════
print(f'Phase 1: Training {len(LAMBDA_VALUES)*len(SEEDS)} runs...')
for lam in LAMBDA_VALUES:
    for seed in SEEDS:
        train_one(lam, seed, synth_train, synth_val, tokenizer)

# ══════════════════════════════════════════════════════════════
# Phase 2: Evaluate all runs on full test set
# ══════════════════════════════════════════════════════════════
print(f'\n\n{"="*70}')
print(f'Phase 2: Evaluating all runs on full test set ({len(synth_test)} eqs)...')
print(f'{"="*70}')
all_metrics = []
for i_run, (lam, seed) in enumerate(
    [(l, s) for l in LAMBDA_VALUES for s in SEEDS]
):
    run_tag = make_run_tag(lam, seed)
    t0 = time.time()
    try:
        m = eval_one(lam, seed, synth_test, tokenizer)
        elapsed = time.time() - t0
        print(f'  [{i_run+1}/{len(LAMBDA_VALUES)*len(SEEDS)}] {run_tag}: '
              f'exact={m["greedy_exact_match"]*100:.1f}% | '
              f'equiv={m["greedy_algebraic_equiv"]*100:.1f}% | '
              f'R²>.9={m["greedy_r2_above_0.9"]*100:.1f}% | '
              f'{elapsed:.0f}s')
        all_metrics.append(m)
    except Exception as e:
        elapsed = time.time() - t0
        print(f'  [{i_run+1}/{len(LAMBDA_VALUES)*len(SEEDS)}] {run_tag}: '
              f'FAILED after {elapsed:.0f}s — {e}')
        all_metrics.append({
            'lambda': lam, 'seed': seed, 'run_tag': run_tag,
            'greedy_exact_match': 0, 'greedy_algebraic_equiv': 0,
            'greedy_r2_above_0.9': 0, 'best_val_loss': float('inf'),
            'best_val_acc': 0, 'greedy_token_acc': 0,
            'n_parseable': 0, 'n_total': 0,
        })

    gc.collect()
    time.sleep(3)

# ── Summary table ──
print(f'\n{"="*70}')
print(f'Sweep complete — {VERSION_TAG} (n_test={len(synth_test)})')
print(f'{"="*70}')
print(f'\n{"λ":>6} {"seed":>6} {"val_loss":>10} {"val_acc":>10} {"exact":>8} {"equiv":>8} {"R²>.9":>8}')
print('-' * 64)
for m in all_metrics:
    print(f'{m["lambda"]:>6.2f} {m["seed"]:>6} {m["best_val_loss"]:>10.4f} {m.get("best_val_acc",0)*100:>9.1f}% '
          f'{m["greedy_exact_match"]*100:>7.1f}% {m["greedy_algebraic_equiv"]*100:>7.1f}% '
          f'{m["greedy_r2_above_0.9"]*100:>7.1f}%')

# ── Averages per lambda ──
print(f'\n{"λ":>6} {"avg_exact":>10} {"avg_equiv":>10} {"avg_R²>.9":>10} {"n_seeds":>8}')
print('-' * 50)
for lam in LAMBDA_VALUES:
    lam_runs = [m for m in all_metrics if m['lambda'] == lam]
    avg_exact = np.mean([m['greedy_exact_match'] for m in lam_runs])
    avg_equiv = np.mean([m['greedy_algebraic_equiv'] for m in lam_runs])
    avg_r2 = np.mean([m['greedy_r2_above_0.9'] for m in lam_runs])
    print(f'{lam:>6.2f} {avg_exact*100:>9.1f}% {avg_equiv*100:>9.1f}% {avg_r2*100:>9.1f}% {len(lam_runs):>8}')

## Training curves

In [ ]:
import matplotlib.pyplot as plt

# Load histories from checkpoints
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for lam in LAMBDA_VALUES:
    for seed in SEEDS:
        run_dir = f'{CKPT_DIR}/{VERSION_TAG}/lam{lam}_seed{seed}'
        ckpt_path = f'{run_dir}/latest.pt'
        if not os.path.exists(ckpt_path):
            continue
        ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        h = ck['history']
        label = f'λ={lam} s={seed}'
        alpha = 0.4 if len(SEEDS) > 1 else 1.0

        axes[0].plot(h['train'], label=label, alpha=alpha)
        if h['val']:
            epochs = list(range(VAL_EVERY, len(h['train']) + 1, VAL_EVERY))[:len(h['val'])]
            axes[1].plot(epochs, h['val'], label=label, alpha=alpha)
        if h.get('val_acc'):
            epochs = list(range(VAL_EVERY, len(h['train']) + 1, VAL_EVERY))[:len(h['val_acc'])]
            axes[2].plot(epochs, [a * 100 for a in h['val_acc']], label=label, alpha=alpha)

axes[0].set_title('Train Loss'); axes[0].set_xlabel('epoch'); axes[0].grid(alpha=0.3)
axes[1].set_title('Val Loss'); axes[1].set_xlabel('epoch'); axes[1].grid(alpha=0.3)
axes[2].set_title('Val Token Acc %'); axes[2].set_xlabel('epoch'); axes[2].grid(alpha=0.3)
axes[0].legend(fontsize=7); axes[1].legend(fontsize=7); axes[2].legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

## Inspect predictions (post-sweep)

Load from metrics.json (saved during sweep) — no model needed.

In [ ]:
def to_infix(prefix_str):
    """Convert prefix string to infix via SymPy. Returns (infix_str, error_msg)."""
    try:
        expr, constants = prefix_to_sympy(prefix_str)
        return str(expr), None
    except Exception as e:
        return None, str(e)

def inspect_run(run_tag, n_show=20):
    """Load metrics.json for a run and display predictions."""
    metrics_path = f'{CKPT_DIR}/{VERSION_TAG}/{run_tag}/metrics.json'
    if not os.path.exists(metrics_path):
        print(f'{run_tag}: no metrics.json found')
        return None
    with open(metrics_path) as f:
        metrics = json.load(f)

    details = metrics.get('details', [])
    if not details:
        print(f'{run_tag}: no prediction details saved')
        return metrics

    print(f'\n{"="*80}')
    print(f'{run_tag} | exact={metrics["greedy_exact_match"]*100:.1f}% | equiv={metrics["greedy_algebraic_equiv"]*100:.1f}%')
    print(f'{"="*80}')

    for i, d in enumerate(details[:n_show]):
        gt_infix, _ = to_infix(d['gt'])
        pred_infix, pred_err = to_infix(d['pred'])

        r2_str = f'{d["r2"]:.4f}' if d['r2'] is not None else 'N/A'
        parseable = d.get('parseable', pred_err is None)
        equiv = d.get('equiv', False)
        if d['exact']:
            status = 'EXACT'
        elif equiv:
            status = 'EQUIV'
        elif parseable:
            status = 'PARSEABLE'
        else:
            status = 'UNPARSEABLE'

        print(f'\n── [{i}] {status} | R²={r2_str} ──')
        print(f'  GT:   {gt_infix}')
        if pred_err is None:
            print(f'  Pred: {pred_infix}')
        else:
            print(f'  Pred: PARSE FAILED: {pred_err}')
            print(f'        {d["pred"]}')

    total = len(details)
    n_unparseable = sum(1 for d in details if not d.get('parseable', True))
    n_exact = sum(1 for d in details if d['exact'])
    n_equiv = sum(1 for d in details if d.get('equiv', False))
    print(f'\n  Total: {total} | Exact: {n_exact} | Equiv: {n_equiv} | Unparseable: {n_unparseable}')

In [ ]:
# Inspect predictions for each (lambda, seed) — pick first seed per lambda for brevity
for lam in LAMBDA_VALUES:
    inspect_run(f'lam{lam}_seed{SEEDS[0]}', n_show=10)

## R² diagnostics: baseline (λ=0) vs JEPA (λ=0.03)

Paired per-equation comparison: does JEPA shift the R² distribution?

In [ ]:
import matplotlib.pyplot as plt

# ── Config ──
LAM_BASE = 0       # baseline
LAM_JEPA = 0.03    # JEPA condition to compare

def _load_r2_vec(lam, seed):
    """Load per-equation R² from metrics.json. Returns list aligned with test set."""
    path = f'{CKPT_DIR}/{VERSION_TAG}/lam{lam}_seed{seed}/metrics.json'
    if not os.path.exists(path):
        return None
    with open(path) as f:
        m = json.load(f)
    return [d.get('r2', None) for d in m.get('details', [])]

def _classify(r2):
    if r2 is None or not np.isfinite(r2) or r2 < 0:
        return 'bad'
    if r2 < 0.9:
        return 'bad'
    if r2 < 0.999:
        return 'close'
    return 'equiv'

bin_edges = ['failed/<0', '0-0.9', '0.9-0.99', '0.99-0.999', '>=0.999']

def _bin_r2(r2):
    if r2 is None or not np.isfinite(r2) or r2 < 0:
        return 'failed/<0'
    if r2 < 0.9:
        return '0-0.9'
    if r2 < 0.99:
        return '0.9-0.99'
    if r2 < 0.999:
        return '0.99-0.999'
    return '>=0.999'


# ════════════════════════════════════════════════════════════
# 1. R² histogram (aggregated across seeds)
# ════════════════════════════════════════════════════════════
counts_base = {b: 0 for b in bin_edges}
counts_jepa = {b: 0 for b in bin_edges}
n_seeds_used = 0

for seed in SEEDS:
    r2_base = _load_r2_vec(LAM_BASE, seed)
    r2_jepa = _load_r2_vec(LAM_JEPA, seed)
    if r2_base is None or r2_jepa is None:
        continue
    n_seeds_used += 1
    for r in r2_base:
        counts_base[_bin_r2(r)] += 1
    for r in r2_jepa:
        counts_jepa[_bin_r2(r)] += 1

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(bin_edges))
w = 0.35
ax.bar(x - w/2, [counts_base[b] for b in bin_edges], w, label=f'λ={LAM_BASE}', alpha=0.8)
ax.bar(x + w/2, [counts_jepa[b] for b in bin_edges], w, label=f'λ={LAM_JEPA}', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(bin_edges, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('# equations (summed over seeds)')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.set_title(f'Held-out R² distribution: λ={LAM_BASE} vs λ={LAM_JEPA}  ({n_seeds_used} seeds)')
plt.tight_layout()
plt.show()


# ════════════════════════════════════════════════════════════
# 2. R² transition table (aggregated across seeds)
# ════════════════════════════════════════════════════════════
classes = ['bad', 'close', 'equiv']
class_labels = {'bad': 'R²<0.9', 'close': '0.9≤R²<0.999', 'equiv': 'R²≥0.999'}

agg_table = np.zeros((3, 3), dtype=int)

for seed in SEEDS:
    r2_base = _load_r2_vec(LAM_BASE, seed)
    r2_jepa = _load_r2_vec(LAM_JEPA, seed)
    if r2_base is None or r2_jepa is None:
        print(f'seed={seed}: missing data, skipping')
        continue
    for rb, rj in zip(r2_base, r2_jepa):
        ci = classes.index(_classify(rb))
        cj = classes.index(_classify(rj))
        agg_table[ci, cj] += 1

n_total = agg_table.sum()
print(f'Transition table (rows=λ={LAM_BASE}, cols=λ={LAM_JEPA}, {n_seeds_used} seeds, n={n_total})')
print()
header = ''.join(f'{class_labels[c]:>16}' for c in classes) + f'{"total":>10}'
print(f'{"":>16}{header}')
for i, rc in enumerate(classes):
    row = ''.join(f'{agg_table[i,j]:>16}' for j in range(3))
    print(f'{class_labels[rc]:>16}{row}{agg_table[i,:].sum():>10}')
col_totals = ''.join(f'{agg_table[:,j].sum():>16}' for j in range(3))
print(f'{"total":>16}{col_totals}{n_total:>10}')

# Key directional counts
print(f'\nKey transitions:')
transitions = [
    ('bad -> close',    agg_table[0, 1]),
    ('bad -> equiv',    agg_table[0, 2]),
    ('close -> bad',    agg_table[1, 0]),
    ('close -> equiv',  agg_table[1, 2]),
    ('equiv -> close',  agg_table[2, 1]),
    ('equiv -> bad',    agg_table[2, 0]),
]
for label, count in transitions:
    pct = count / max(n_total, 1) * 100
    print(f'  {label:>20}: {count:>5}  ({pct:.1f}%)')

net_up = agg_table[0,1] + agg_table[0,2] + agg_table[1,2]
net_down = agg_table[1,0] + agg_table[2,0] + agg_table[2,1]
print(f'\n  Net improved (any upward):  {net_up}')
print(f'  Net degraded (any downward): {net_down}')
print(f'  Net change: {net_up - net_down:+d}')

In [ ]:
# ════════════════════════════════════════════════════════════
# 3. Example equations from each transition category
# ════════════════════════════════════════════════════════════
N_EXAMPLES = 3  # examples per transition per seed

def _load_details(lam, seed):
    path = f'{CKPT_DIR}/{VERSION_TAG}/lam{lam}_seed{seed}/metrics.json'
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f).get('details', [])

def _r2_str(r2):
    if r2 is None:
        return 'N/A'
    return f'{r2:.4f}' if np.isfinite(r2) else 'N/A'

# Collect examples keyed by transition type
transition_examples = {t: [] for t in [
    'bad->close', 'bad->equiv', 'close->bad',
    'close->equiv', 'equiv->close', 'equiv->bad',
]}

for seed in SEEDS:
    det_base = _load_details(LAM_BASE, seed)
    det_jepa = _load_details(LAM_JEPA, seed)
    if det_base is None or det_jepa is None:
        continue

    for i, (db, dj) in enumerate(zip(det_base, det_jepa)):
        cb = _classify(db.get('r2'))
        cj = _classify(dj.get('r2'))
        if cb == cj:
            continue
        key = f'{cb}->{cj}'
        if key in transition_examples:
            gt_infix, _ = to_infix(db['gt'])
            base_infix, _ = to_infix(db['pred'])
            jepa_infix, _ = to_infix(dj['pred'])
            transition_examples[key].append({
                'seed': seed, 'idx': i,
                'gt': gt_infix or db['gt'],
                'base_pred': base_infix or db['pred'],
                'jepa_pred': jepa_infix or dj['pred'],
                'r2_base': db.get('r2'),
                'r2_jepa': dj.get('r2'),
            })

for trans_key, examples in transition_examples.items():
    if not examples:
        continue
    print(f'\n{"="*70}')
    print(f'  {trans_key}  ({len(examples)} total, showing {min(N_EXAMPLES, len(examples))})')
    print(f'{"="*70}')
    for ex in examples[:N_EXAMPLES]:
        print(f'\n  [seed={ex["seed"]}, eq #{ex["idx"]}]')
        print(f'    GT:        {ex["gt"]}')
        print(f'    base pred: {ex["base_pred"]}  (R²={_r2_str(ex["r2_base"])})')
        print(f'    JEPA pred: {ex["jepa_pred"]}  (R²={_r2_str(ex["r2_jepa"])})')

## Pretraining diagnostics & summary

In [ ]:
# ── Stage 1 diagnostics (pretrain mode only) ──
# JEPA loss and CE loss are different objectives on different scales, so they
# get separate axes rather than being overlaid.
_pre = [(lam, seed, torch.load(
            f'{CKPT_DIR}/{VERSION_TAG}/{make_run_tag(lam, seed)}/latest.pt',
            map_location='cpu', weights_only=False))
        for lam in LAMBDA_VALUES for seed in SEEDS
        if os.path.exists(
            f'{CKPT_DIR}/{VERSION_TAG}/{make_run_tag(lam, seed)}/latest.pt')]
_pre = [(l, s, ck) for l, s, ck in _pre if ck.get('history', {}).get('pretrain_jepa')]

if not _pre:
    print('No pretrain runs found — set TRAINING_MODE = "pretrain" and train first.')
else:
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    for lam, seed, ck in _pre:
        h = ck['history']
        pe = range(1, len(h['pretrain_jepa']) + 1)
        lbl = f'λ={lam} s={seed}'
        ax[0].plot(pe, h['pretrain_jepa'], marker='o', label=lbl)
        ax[1].plot(pe, h['pretrain_std_num'], marker='o', label=f'{lbl} z_num')
        ax[1].plot(pe, h['pretrain_std_sym'], marker='s', ls='--',
                   label=f'{lbl} z_sym')
        if h.get('val_acc'):
            ax[2].plot(range(1, len(h['val_acc']) + 1),
                       [a * 100 for a in h['val_acc']], label=lbl)

    ax[0].set_title('Stage 1: JEPA loss'); ax[0].set_xlabel('pretrain epoch')
    ax[1].set_title('Stage 1: embedding std (→0 = collapse)')
    ax[1].set_xlabel('pretrain epoch')
    ax[2].set_title('Stage 2: val token acc %'); ax[2].set_xlabel('finetune epoch')
    for a in ax:
        a.grid(alpha=0.3); a.legend(fontsize=7)
    plt.tight_layout(); plt.show()

In [ ]:
# ── Experiment summary ──
def summarise(lam, seed):
    tag = make_run_tag(lam, seed)
    d = f'{CKPT_DIR}/{VERSION_TAG}/{tag}'
    ck = (torch.load(f'{d}/latest.pt', map_location='cpu', weights_only=False)
          if os.path.exists(f'{d}/latest.pt') else None)
    m = json.load(open(f'{d}/metrics.json')) if os.path.exists(f'{d}/metrics.json') else None
    if ck is None and m is None:
        print(f'{tag}: nothing saved yet'); return
    h = (ck or {}).get('history', {})
    mode = (m or ck).get('training_mode', 'joint')
    pj = h.get('pretrain_jepa') or []

    print(f'\n{"="*54}\n{tag}\n{"="*54}')
    print(f'  Training mode          : {mode}')
    print(f'  Seed                   : {seed}')
    print(f'  Lambda                 : {lam}'
          f'{"  (unused in stage 1)" if mode == "pretrain" else ""}')
    if mode == 'pretrain':
        print(f'  JEPA pretrain epochs   : {(m or ck).get("jepa_pretrain_epochs", "?")}')
        print(f'  Pretrain stopgrad      : {(m or ck).get("pretrain_stopgrad", "?")}')
        print(f'  Final JEPA pretrain    : {pj[-1]:.4f}' if pj else
              '  Final JEPA pretrain    : N/A')
        if h.get('pretrain_std_num'):
            print(f'  Final std(z_num)       : {h["pretrain_std_num"][-1]:.4f}'
                  f'   std(z_sym): {h["pretrain_std_sym"][-1]:.4f}')
    else:
        print('  JEPA pretrain epochs   : N/A (joint mode)')
    print(f'  Fine-tuning epochs     : {ck["epoch"] if ck else "?"}')
    if ck:
        print(f'  Best val loss          : {ck.get("best_val", float("nan")):.4f}')
        print(f'  Best val token acc     : {ck.get("best_val_acc", 0)*100:.2f}%')
    if m:
        print(f'  Exact                  : {m["greedy_exact_match"]*100:.1f}%')
        print(f'  Equivalent             : {m["greedy_algebraic_equiv"]*100:.1f}%')
        print(f'  R² > 0.9               : {m["greedy_r2_above_0.9"]*100:.1f}%')
    else:
        print('  (test metrics: run eval_one / Phase 2 first)')

for _lam in LAMBDA_VALUES:
    for _seed in SEEDS:
        summarise(_lam, _seed)